# Лабораторная работа №4
## Применение методов машинного обучения в решениях профильных задач бизнеса
### Студент: Шевченко Ю.С., ИБМ 3-63Б

### Ячейка 1: Загрузка данных и расчет целевой переменной

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
import warnings
warnings.filterwarnings('ignore')

# 1. Загрузка данных
df = pd.read_csv('../data/ownership_data.csv')

# 2. Расчет дельты по компаниям
df['DeltaForeign'] = df.groupby('Company')['ForeignShare'].diff().abs()
df['DeltaForeign'] = df['DeltaForeign'].fillna(0)

# 3. Целевая переменная: 1 если скачок >= 20 п.п., иначе 0
df['HighOwnershipVolatility'] = (df['DeltaForeign'] >= 20).astype(int)

# Удаляем 2013 год (нет предыдущего значения для дельты)
df = df[df['Year'] > 2013].copy()

print('Размер датасета после очистки:', df.shape)
print('\nРаспределение целевой переменной:')
print(df['HighOwnershipVolatility'].value_counts())
df.head()

### Ячейка 2: EDA - Визуализация данных

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Баланс классов
sns.countplot(x='HighOwnershipVolatility', data=df, palette='Set2', ax=axes[0, 0])
axes[0, 0].set_title('Распределение целевой переменной (0=стабильно, 1=скачок)')

# 2. Распределение доли иностранных инвесторов
sns.histplot(data=df, x='ForeignShare', kde=True, ax=axes[0, 1], color='skyblue')
axes[0, 1].set_title('Распределение доли иностранных инвесторов')

# 3. DeltaForeign по компаниям
sns.boxplot(data=df, x='Company', y='DeltaForeign', ax=axes[1, 0], palette='pastel')
axes[1, 0].set_title('Величина изменений доли иностранных инвесторов по компаниям')
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Корреляционная матрица (для числовых признаков)
corr_cols = ['ForeignShare', 'DeltaForeign', 'HighOwnershipVolatility']
sns.heatmap(df[corr_cols].corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1, ax=axes[1, 1])
axes[1, 1].set_title('Корреляционная матрица')

plt.tight_layout()
plt.show()

### Ячейка 3: Предобработка данных (Preprocessing)

In [ ]:
# Подготовка признаков (X) и целевой переменной (y)
# Удаляем лишние колонки: Year, Company, целевую
X = df.drop(columns=['HighOwnershipVolatility', 'Year', 'Company'])
y = df['HighOwnershipVolatility']

# Настройка предобработки
# Числовые признаки масштабируем, категориальные кодируем
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['ForeignShare', 'DeltaForeign']),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), ['Sector'])
    ]
)

X_proc = preprocessor.fit_transform(X)

# Разделение на тренировочную и тестовую выборки (75/25)
X_train, X_test, y_train, y_test = train_test_split(
    X_proc, y, test_size=0.25, random_state=42, stratify=y
)

print('Размер Train:', X_train.shape[0])
print('Размер Test:', X_test.shape[0])

### Ячейка 4: Обучение моделей

In [ ]:
# Инициализация моделей (используем class_weight='balanced' из-за дисбаланса)
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42)
}

results = {}
print('Обучение моделей...\n')

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # Расчет метрик
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    }
    print(f"{name}:")
    print(f"  Accuracy : {results[name]['Accuracy']:.3f}")
    print(f"  F1-score : {results[name]['F1']:.3f}")
    print(f"  ROC-AUC  : {results[name]['ROC-AUC']:.3f}\n")

### Ячейка 5: Оценка качества (Confusion Matrix & ROC)

In [ ]:
# Выбор лучшей модели по F1-score
best_name = max(results, key=lambda x: results[x]['F1'])
print(f'>> Лучшая модель: {best_name}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1. Матрица ошибок
cm = confusion_matrix(y_test, results[best_name]['y_pred'])
ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap='Blues', ax=axes[0])
axes[0].set_title(f'Confusion Matrix\n({best_name})')

# 2. ROC-кривая
RocCurveDisplay.from_estimator(results[best_name]['model'], X_test, y_test, ax=axes[1])
axes[1].set_title('ROC Curve')

plt.tight_layout()
plt.show()

### Ячейка 6: Важность признаков

In [ ]:
# Анализ важности признаков для лучшей модели
feat_names = preprocessor.get_feature_names_out()
# Для случайного леса берем feature_importances_, для логистической регрессии - coef_
if hasattr(results[best_name]['model'], 'feature_importances_'):
    importances = results[best_name]['model'].feature_importances_
else:
    importances = results[best_name]['model'].coef_[0]

feat_imp = pd.Series(importances, index=feat_names).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
feat_imp.plot(kind='barh', color='skyblue', edgecolor='black')
plt.title(f'Важность признаков\n({best_name})', fontsize=14, fontweight='bold')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()

print('\nТоп-3 важных признака:')
for i, (feat, imp) in enumerate(feat_imp.head(3).items(), 1):
    print(f"{i}. {feat}: {imp:.3f}")